# Evaluation 

### _Create end-to-end example of customer service assistant:_

Steps to Build Customer Service assistant:
1. Use Moderation API for input to see if it is flagged
2. If the input does not flag, extract products 
3. If products are listed, look them in product information
4. Answer users question with the Model
5. Check answer with Moderation API and return it to the user if it is not flagged

Use panel python package for chatbot GUI:

    import os
    import openai
    import sys
    sys.path.append('../..')
    import utils

    import panel as pn  # GUI
    pn.extension()

    from dotenv import load_dotenv, find_dotenv
    _ = load_dotenv(find_dotenv()) # read local .env file

    openai.api_key  = os.environ['OPENAI_API_KEY']

    def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0, max_tokens=500):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, 
        max_tokens=max_tokens, 
    )
    return response.choices[0].message["content"]

### _Process User Message:_

It is a helper function that takes in user input, all previous messages to the model after processing user input.

Steps to process user message:
1. Use Moderation API for input to see if it is flagged
2. If the input does not flag, extract list of products 
3. If products are listed, look them in product information
4. Generate response to the user's question with the Model
5. Check response with Moderation API and return it to the user if it is not flagged
6. Ask model is the generated response answers the user's question well
7. If the model has responded well based on evaluation, return generated response and all previous messages between user and assistant

    def process_user_message(user_input, all_messages, debug=True):
        delimiter = "```"
        
        # Step 1: Check input to see if it flags the Moderation API or is a prompt injection
        response = openai.Moderation.create(input=user_input)
        moderation_output = response["results"][0]

        if moderation_output["flagged"]:
            print("Step 1: Input flagged by Moderation API.")
            return "Sorry, we cannot process this request."

        if debug: print("Step 1: Input passed moderation check.")
        
        category_and_product_response = utils.find_category_and_product_only(user_input, utils.get_products_and_category())
        #print(print(category_and_product_response)
        # Step 2: Extract the list of products
        category_and_product_list = utils.read_string_to_list(category_and_product_response)
        #print(category_and_product_list)

        if debug: print("Step 2: Extracted list of products.")

        # Step 3: If products are found, look them up
        product_information = utils.generate_output_string(category_and_product_list)
        if debug: print("Step 3: Looked up product information.")

        # Step 4: Answer the user question
        system_message = f"""
        You are a customer service assistant for a large electronic store. \
        Respond in a friendly and helpful tone, with concise answers. \
        Make sure to ask the user relevant follow-up questions.
        """
        messages = [
            {'role': 'system', 'content': system_message},
            {'role': 'user', 'content': f"{delimiter}{user_input}{delimiter}"},
            {'role': 'assistant', 'content': f"Relevant product information:\n{product_information}"}
        ]

        final_response = get_completion_from_messages(all_messages + messages)
        if debug:print("Step 4: Generated response to user question.")
        all_messages = all_messages + messages[1:]

        # Step 5: Put the answer through the Moderation API
        response = openai.Moderation.create(input=final_response)
        moderation_output = response["results"][0]

        if moderation_output["flagged"]:
            if debug: print("Step 5: Response flagged by Moderation API.")
            return "Sorry, we cannot provide this information."

        if debug: print("Step 5: Response passed moderation check.")

        # Step 6: Ask the model if the response answers the initial user query well
        user_message = f"""
        Customer message: {delimiter}{user_input}{delimiter}
        Agent response: {delimiter}{final_response}{delimiter}

        Does the response sufficiently answer the question?
        """
        messages = [
            {'role': 'system', 'content': system_message},
            {'role': 'user', 'content': user_message}
        ]
        evaluation_response = get_completion_from_messages(messages)
        if debug: print("Step 6: Model evaluated the response.")

        # Step 7: If yes, use this answer; if not, say that you will connect the user to a human
        if "Y" in evaluation_response:  # Using "in" instead of "==" to be safer for model output variation (e.g., "Y." or "Yes")
            if debug: print("Step 7: Model approved the response.")
            return final_response, all_messages
        else:
            if debug: print("Step 7: Model disapproved the response.")
            neg_str = "I'm unable to provide the information you're looking for. I'll connect you with a human representative for further assistance."
            return neg_str, all_messages

    user_input = "tell me about the smartx pro phone and the fotosnap camera, the dslr one. Also what tell me about your tvs"
    response,_ = process_user_message(user_input,[])
    print(response)

#### _Example for Processing user message:_

Test:

        user_input = "tell me about the smartx pro phone and the fotosnap camera, the dslr one. Also what tell me about your tvs"
        response,_ = process_user_message(user_input,[])
        print(response)

Response:

    Step 1: Input passed moderation check.
    Step 2: Extracted list of products.
    Step 3: Looked up product information.
    Step 4: Generated response to user question.
    Step 5: Response passed moderation check.
    Step 6: Model evaluated the response.
    Step 7: Model approved the response.

    The SmartX Pro phone is a high-performance smartphone with a powerful processor, advanced camera features, and a sleek design. It offers a smooth user experience and great connectivity options.

    The FotoSnap camera is a DSLR camera known for its professional-grade image quality, interchangeable lenses, and advanced shooting modes. It's perfect for photography enthusiasts and professionals looking to capture stunning photos.

    As for our TVs, we offer a wide range of options including LED, OLED, and QLED TVs in various sizes and resolutions. Our TVs are known for their vibrant colors, sharp image quality, and smart features like built-in streaming services and voice control.

    Do you have any specific questions about the SmartX Pro phone, FotoSnap camera, or our TVs? Are you looking for any particular features or specifications in these products?

### _Build Chatbot GUI:_

_Function that collects user and assistant messages over time:_

    def collect_messages(debug=False):
        user_input = inp.value_input
        if debug: print(f"User Input = {user_input}")
        if user_input == "":
            return
        inp.value = ''
        global context
        #response, context = process_user_message(user_input, context, utils.get_products_and_category(),debug=True)
        response, context = process_user_message(user_input, context, debug=False)
        context.append({'role':'assistant', 'content':f"{response}"})
        panels.append(
            pn.Row('User:', pn.pane.Markdown(user_input, width=600)))
        panels.append(
            pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))
    
        return pn.Column(*panels)

_Display Chatbot UI:_

    panels = [] # collect display 

    context = [ {'role':'system', 'content':"You are Service Assistant"} ]  

    inp = pn.widgets.TextInput( placeholder='Enter text here…')
    button_conversation = pn.widgets.Button(name="Service Assistant")

    interactive_conversation = pn.bind(collect_messages, button_conversation)

    dashboard = pn.Column(
        inp,
        pn.Row(button_conversation),
        pn.panel(interactive_conversation, loading_indicator=True, height=300),
    )

    dashboard

## Evaluation Part 1

Best Practices for evaluating LLM outputs:
- Tune prompts on handful of examples
- Add tricky examples to the model for training
- Regression testing to verify model still works on previous test cases after changing prompt
- develop metrics to measure performance 
- collect and use a test set as examples

_Get the relevant products and categories:_    

    products_and_category = utils.get_products_and_category()


_Find relevant product and category names:_

    def find_category_and_product_v1(user_input,products_and_category):

        delimiter = "####"
        system_message = f"""
        You will be provided with customer service queries. \
        The customer service query will be delimited with {delimiter} characters.
        Output a python list of json objects, where each object has the following format:
            'category': <one of Computers and Laptops, Smartphones and Accessories, Televisions and Home Theater Systems, \
        Gaming Consoles and Accessories, Audio Equipment, Cameras and Camcorders>,
        AND
            'products': <a list of products that must be found in the allowed products below>


        Where the categories and products must be found in the customer service query.
        If a product is mentioned, it must be associated with the correct category in the allowed products list below.
        If no products or categories are found, output an empty list.
        

        List out all products that are relevant to the customer service query based on how closely it relates
        to the product name and product category.
        Do not assume, from the name of the product, any features or attributes such as relative quality or price.

        The allowed products are provided in JSON format.
        The keys of each item represent the category.
        The values of each item is a list of products that are within that category.
        Allowed products: {products_and_category}
        

        """

_Example for model response:_

        few_shot_user_1 = """I want the most expensive computer."""
        few_shot_assistant_1 = """ 
        [{'category': 'Computers and Laptops', \
    'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
        """
        
        messages =  [  
        {'role':'system', 'content': system_message},    
        {'role':'user', 'content': f"{delimiter}{few_shot_user_1}{delimiter}"},  
        {'role':'assistant', 'content': few_shot_assistant_1 },
        {'role':'user', 'content': f"{delimiter}{user_input}{delimiter}"},  
        ] 
        return get_completion_from_messages(messages)


_Evaluate some queries:_

1. User message example 1 -

        customer_msg_0 = f"""Which TV can I buy if I'm on a budget?"""
        products_by_category_0 = find_category_and_product_v1(customer_msg_0,
                                                            products_and_category)
        print(products_by_category_0)

    Response - 

        [{'category': 'Televisions and Home Theater Systems', 'products': ['CineView 4K TV', 'SoundMax Home Theater', 'CineView 8K TV', 'SoundMax Soundbar', 'CineView OLED TV']}]


2. User message example 2 -

        customer_msg_1 = f"""I need a charger for my smartphone"""
        products_by_category_1 = find_category_and_product_v1(customer_msg_1,
                                                            products_and_category)
        print(products_by_category_1)

    Response -

        [{'category': 'Smartphones and Accessories', 'products': ['MobiTech Wireless Charger']}]

2. User message example 3 -

        customer_msg_3 = f"""
        tell me about the smartx pro phone and the fotosnap camera, the dslr one.
        Also, what TVs do you have?"""
        products_by_category_3 = find_category_and_product_v1(customer_msg_3,
                                                            products_and_category)
        print(products_by_category_3)

    Response -

        [{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone']}, {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera']}]

_Modify prompt to work on hard test cases:_

        def find_category_and_product_v2(user_input,products_and_category):
            """
            Added: Do not output any additional text that is not in JSON format.
            Added a second example (for few-shot prompting) where user asks for 
            the cheapest computer. In both few-shot examples, the shown response 
            is the full list of products in JSON only.
            """
            delimiter = "####"
            system_message = f"""
            You will be provided with customer service queries. \
            The customer service query will be delimited with {delimiter} characters.
            Output a python list of json objects, where each object has the following format:
                'category': <one of Computers and Laptops, Smartphones and Accessories, Televisions and Home Theater Systems, \
            Gaming Consoles and Accessories, Audio Equipment, Cameras and Camcorders>,
            AND
                'products': <a list of products that must be found in the allowed products below>
            Do not output any additional text that is not in JSON format.
            Do not write any explanatory text after outputting the requested JSON.


            Where the categories and products must be found in the customer service query.
            If a product is mentioned, it must be associated with the correct category in the allowed products list below.
            If no products or categories are found, output an empty list.
            

            List out all products that are relevant to the customer service query based on how closely it relates
            to the product name and product category.
            Do not assume, from the name of the product, any features or attributes such as relative quality or price.

            The allowed products are provided in JSON format.
            The keys of each item represent the category.
            The values of each item is a list of products that are within that category.
            Allowed products: {products_and_category}
            

            """
            
            few_shot_user_1 = """I want the most expensive computer. What do you recommend?"""
            few_shot_assistant_1 = """ 
            [{'category': 'Computers and Laptops', \
        'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
            """
            
            few_shot_user_2 = """I want the most cheapest computer. What do you recommend?"""
            few_shot_assistant_2 = """ 
            [{'category': 'Computers and Laptops', \
        'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
            """

Train on few example test cases -

            messages =  [  
            {'role':'system', 'content': system_message},    
            {'role':'user', 'content': f"{delimiter}{few_shot_user_1}{delimiter}"},  
            {'role':'assistant', 'content': few_shot_assistant_1 },
            {'role':'user', 'content': f"{delimiter}{few_shot_user_2}{delimiter}"},  
            {'role':'assistant', 'content': few_shot_assistant_2 },
            {'role':'user', 'content': f"{delimiter}{user_input}{delimiter}"},  
            ] 
            return get_completion_from_messages(messages)


_Evaluate test cases by comparing to the ideal answers:_

        import json
        def eval_response_with_ideal(response,
                                    ideal,
                                    debug=False):
            
            if debug:
                print("response")
                print(response)
            
            # json.loads() expects double quotes, not single quotes
            json_like_str = response.replace("'",'"')
            
            # parse into a list of dictionaries
            l_of_d = json.loads(json_like_str)
            
            # special case when response is empty list
            if l_of_d == [] and ideal == []:
                return 1
            
            # otherwise, response is empty 
            # or ideal should be empty, there's a mismatch
            elif l_of_d == [] or ideal == []:
                return 0
            
            correct = 0    
            
            if debug:
                print("l_of_d is")
                print(l_of_d)
            for d in l_of_d:

                cat = d.get('category')
                prod_l = d.get('products')
                if cat and prod_l:
                    # convert list to set for comparison
                    prod_set = set(prod_l)
                    # get ideal set of products
                    ideal_cat = ideal.get(cat)
                    if ideal_cat:
                        prod_set_ideal = set(ideal.get(cat))
                    else:
                        if debug:
                            print(f"did not find category {cat} in ideal")
                            print(f"ideal: {ideal}")
                        continue
                        
                    if debug:
                        print("prod_set\n",prod_set)
                        print()
                        print("prod_set_ideal\n",prod_set_ideal)

                    if prod_set == prod_set_ideal:
                        if debug:
                            print("correct")
                        correct +=1
                    else:
                        print("incorrect")
                        print(f"prod_set: {prod_set}")
                        print(f"prod_set_ideal: {prod_set_ideal}")
                        if prod_set <= prod_set_ideal:
                            print("response is a subset of the ideal answer")
                        elif prod_set >= prod_set_ideal:
                            print("response is a superset of the ideal answer")

            # count correct over total number of items in list
            pc_correct = correct / len(l_of_d)
                
            return pc_correct

_Evaluation result between generated result and expect result from example Test case:_

Generated Response -

    response = find_category_and_product_v2(msg_ideal_pairs_set[7]["customer_msg"],
                                            products_and_category)
    print(f'Resonse: {response}')

Evaluate Generated Response with expected result -

    eval_response_with_ideal(response,
                                msg_ideal_pairs_set[7]["ideal_answer"])


Evaluation -

    Resonse:  
    [{'category': 'Gaming Consoles and Accessories', 'products': ['GameSphere X', 'ProGamer Controller', 'GameSphere Y', 'ProGamer Racing Wheel', 'GameSphere VR Headset']}]
    
    1.0